# Phase 5: High-Performance Ensemble & Production Pipeline

This notebook covers:
1. **Bagging Architecture**: Random Forest with Bootstrap Sampling
2. **Boosting Suite**: AdaBoost, Gradient Boosting, XGBoost
3. **Stacking Ensemble**: Multi-level Stacking Classifier with Logistic Regression Meta-Learner
4. **Production Lifecycle Execution**: The 9-step automated pipeline
5. **Real-Time Serving API**: Testing FastAPI prediction endpoint

In [ ]:
from mlo.data.generator import generate_customer_dataset
from mlo.data.encoder import CategoricalEncoder
from mlo.data.preprocessor import DataPreprocessor
from mlo.ensemble.bagging import CustomerRandomForest
from mlo.ensemble.boosting import CustomerAdaBoost, CustomerGradientBoosting, CustomerXGBoost
from mlo.ensemble.stacking import CustomerStackingClassifier
from mlo.evaluation.metrics import calculate_classification_metrics
from mlo.pipeline.lifecycle import ProductionPipeline
import pandas as pd

## 1. Train and Compare Ensemble Models

In [ ]:
df = generate_customer_dataset(n_samples=800, random_state=42)
encoder = CategoricalEncoder()
df_encoded = encoder.fit_transform(df)
X = df_encoded[encoder.feature_columns_].values
y = df_encoded['Churn'].values

preproc = DataPreprocessor(scaler_type="standard")
X_train, X_test, y_train, y_test = preproc.split(pd.DataFrame(X), pd.Series(y), test_size=0.2, random_state=42, stratify=pd.Series(y))
X_train_sc = preproc.fit_transform(X_train.values)
X_test_sc = preproc.transform(X_test.values)

models = {
    "Random Forest": CustomerRandomForest(n_estimators=100, max_depth=5),
    "AdaBoost": CustomerAdaBoost(n_estimators=50),
    "Gradient Boosting": CustomerGradientBoosting(n_estimators=80),
    "XGBoost": CustomerXGBoost(n_estimators=80),
    "Stacking Meta-Learner": CustomerStackingClassifier()
}

results = []
for name, model in models.items():
    model.fit(X_train_sc, y_train.values)
    preds = model.predict(X_test_sc)
    m = calculate_classification_metrics(y_test.values, preds)
    results.append({"Model": name, "Accuracy": m["accuracy"], "F1": m["f1"]})
pd.DataFrame(results)

## 2. Execute End-to-End 9-Step Production Pipeline

In [ ]:
pipeline = ProductionPipeline()
pipeline_report = pipeline.run()
print("Exported Pipeline Report Summary:")
print("Regression MSE:", pipeline_report["regression_metrics"]["mse"])
print("Classification Accuracy:", pipeline_report["classification_metrics"]["accuracy"])